# Define volume change areas around validation data transects for Duck

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import numpy as np
import pickle

from shapely import LineString, intersection

from coastal_data import CD_geometry

/home/bene/PhD-git/envs/universe/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Paths

In [2]:
path_to_data_folder = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/'
# path_to_data_folder = '/home/jovyan/data/98_paper2_kf_dtm/'

path_validation = path_to_data_folder + '02_Duck_validation_data/'
path_poly = path_to_data_folder + '22_23_Duck_DEM/output/'
path_output = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/06_Duck_transect_polys/'

## Data

In [3]:
duck_vali = xr.open_dataset(path_validation+'duck_validation.nc')

In [4]:
poly_sl_corr = pickle.load(open(path_poly + 'poly_sl_corr4326.pkl', 'rb'))

## Define transect areas
duck_vali = xr.open_dataset(path_validation+'duck_validation.nc')

In [14]:
bs = CD_geometry.dist_meter_to_dist_deg(12) # buffer size
transect_polys = {}
for idx_id in range(0, len(duck_vali.yFRF)):
    transect = duck_vali.isel(time=0, yFRF=idx_id)
    transect_line = LineString(zip(transect.longitude.values, transect.latitude.values))
    transect_poly = transect_line.buffer(bs)
    # Cut out the part in the target area
    transect_poly_red = intersection(transect_poly, poly_sl_corr)
    transect_polys[idx_id] = transect_poly_red

In [16]:
pickle.dump(transect_polys, open(path_output + 'transect_polys.pkl', 'wb'))